# JORINOVA NEXUS — Intent classifier LoRA fine-tune

Fine-tunes `microsoft/Phi-3-mini-4k-instruct` on the JORINOVA intent
training data (intent.jsonl) using PEFT / LoRA. Designed to run on a
free Colab T4 in ~25–35 minutes per epoch.

## What you need before pressing Run All

1. Runtime → Change runtime type → **T4 GPU** (free tier is enough).
2. The two data files from the JORINOVA repo:
   - `backend/training/golden/intent_golden.jsonl`  (192 hand-curated examples — eval set)
   - `datasets/intent.jsonl`                         (153 synthetic — train + held-out split)
   Either upload them via the left sidebar or git-clone the repo.

## What it does

1. Loads the data, splits 70 / 15 / 15 train / val / test.
2. Wraps Phi-3 with a LoRA adapter (rank 8, ~3 M trainable params).
3. Trains for 2 epochs at 5e-4 with cosine schedule.
4. Scores against the locked golden set + reports a confusion matrix.
5. Exports an Ollama `Modelfile` so you can run the result locally.

## 1. Install + import

In [ ]:
!pip install -q transformers==4.45.0 peft==0.13.0 datasets==3.0.0 accelerate==1.0.0 bitsandbytes==0.44.0 trl==0.11.0

In [ ]:
import json, random, torch
from pathlib import Path
from collections import Counter, defaultdict
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

random.seed(42); torch.manual_seed(42)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2. Load the JORINOVA data

Upload `intent.jsonl` (training) and `intent_golden.jsonl` (eval) to the
Colab files panel, OR clone the repo:

In [ ]:
# Option A: clone from GitHub (replace with your fork if private)
# !git clone https://github.com/jorinova/JORINOVA.git
# DATA_DIR = 'JORINOVA'

# Option B: assume the two files are uploaded next to the notebook
DATA_DIR = '.'
TRAIN_PATH  = f'{DATA_DIR}/intent.jsonl'              # 153 synthetic
GOLDEN_PATH = f'{DATA_DIR}/intent_golden.jsonl'        # 192 locked truth

def jload(p):
    return [json.loads(l) for l in Path(p).read_text(encoding='utf-8').splitlines() if l.strip()]

train_all = jload(TRAIN_PATH)
golden    = jload(GOLDEN_PATH)
print(f'training pool : {len(train_all)} examples')
print(f'golden eval   : {len(golden)} examples')
print('label distribution:', Counter(r['intent'] for r in train_all))

## 3. Format as chat-completion supervised pairs

Phi-3 expects messages in OpenAI-style. We give it a clear classifier
instruction; the target is the bare intent label so token length stays
small.

In [ ]:
SYSTEM = (
    'You are an intent classifier for the JORINOVA NEXUS training assistant. '
    'Output a SINGLE word from this closed set:\n'
    '  start | next | pause | resume | restart | stop\n'
    '  greet | wake | help | thanks | unknown\n'
    'Languages: English, French, Kinyarwanda.'
)

def to_chat(r):
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM},
            {'role': 'user',   'content': f"[{r['language']}] {r['text']}"},
            {'role': 'assistant', 'content': r['intent']},
        ],
    }

random.shuffle(train_all)
split = int(0.85 * len(train_all))
train_ds = Dataset.from_list([to_chat(r) for r in train_all[:split]])
val_ds   = Dataset.from_list([to_chat(r) for r in train_all[split:]])
print(f'train: {len(train_ds)}  val: {len(val_ds)}')

## 4. Load Phi-3 (4-bit quantised) + attach LoRA

In [ ]:
MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto', trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 5. Train

In [ ]:
cfg = SFTConfig(
    output_dir='./out',
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    max_seq_length=256,
    optim='paged_adamw_8bit',
    bf16=True,
    report_to='none',
)
trainer = SFTTrainer(
    model=model, args=cfg,
    train_dataset=train_ds, eval_dataset=val_ds,
    tokenizer=tokenizer,
)
trainer.train()

## 6. Score against the locked golden set

If the fine-tune lands below 95 % on the golden we keep the regex
baseline (which is 100 %) and ship the model only as a fallback for
novel utterances.

In [ ]:
VALID = {'start','next','pause','resume','restart','stop','greet','wake','help','thanks','unknown'}

def predict(text, lang):
    msgs = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user',   'content': f'[{lang}] {text}'},
    ]
    inp = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(inp, max_new_tokens=6, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True).strip().lower()
    # First token from VALID wins
    for tok in raw.replace('.', ' ').split():
        if tok in VALID:
            return tok
    return 'unknown'

correct = 0; misses = []; by_lang = defaultdict(lambda: [0, 0])
for r in golden:
    pred = predict(r['text'], r['language'])
    ok = pred == r['intent']
    by_lang[r['language']][1] += 1
    if ok:
        correct += 1
        by_lang[r['language']][0] += 1
    else:
        misses.append((r['language'], r['text'], r['intent'], pred))

print(f'\nFINE-TUNED accuracy: {correct}/{len(golden)} = {correct/len(golden):.1%}')
for lang, (ok, tot) in by_lang.items():
    print(f'  {lang}: {ok}/{tot} = {ok/tot:.0%}')
print(f'\nmisses ({len(misses)}):')
for lang, t, g, p in misses[:25]:
    print(f'  [{lang}] {t!r}  gold={g}  pred={p}')

## 7. Export as Ollama Modelfile

Merge the LoRA into the base, save in GGUF format (via llama.cpp or the
HF safetensors→GGUF flow), and write a Modelfile you can `ollama create`
locally.

In [ ]:
merged = model.merge_and_unload()
merged.save_pretrained('./jorinova-intent-merged')
tokenizer.save_pretrained('./jorinova-intent-merged')

MODELFILE = '''FROM ./jorinova-intent.gguf
PARAMETER num_ctx 2048
PARAMETER temperature 0
PARAMETER top_p 0.9
SYSTEM You are an intent classifier for the JORINOVA NEXUS training assistant. Output a SINGLE word: start, next, pause, resume, restart, stop, greet, wake, help, thanks, or unknown.
'''
Path('Modelfile').write_text(MODELFILE)
print('Wrote ./jorinova-intent-merged/ and Modelfile')
print()
print('Next steps on your laptop:')
print('  1. Download jorinova-intent-merged/ from Colab files panel')
print('  2. Convert to GGUF (llama.cpp --convert-hf-to-gguf)')
print('  3. ollama create jorinova-intent -f Modelfile')
print('  4. Set OLLAMA_MODEL_CHAT=jorinova-intent in backend/.env')